# Dependências diretas univariadas

## 1. Objetivo

Os métodos recuperam sete lags conhecidos? Comparamos suportes consecutivos e esparsos, com efeitos diretos equilibrados e um limite analítico para efeitos indiretos.

Checkpoint oficial Amazon Chronos-2, sem treinamento. Previsão da mediana de um passo, com entradas observadas para todos os canais e sem valores futuros.

[Fonte do modelo](https://huggingface.co/amazon/chronos-2) · [Referência técnica](https://arxiv.org/abs/2510.15821)

## 2. Preparação do ambiente

Esta célula carrega o checkpoint e prepara a conferência da previsão e dos gradientes. Os cenários abaixo usam o mesmo modelo com pesos congelados.

In [1]:
import json
import platform
from functools import partial
from importlib.metadata import version

import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display
from chronos import Chronos2Pipeline

torch.set_num_threads(4)
revision = "29ec3766d36d6f73f0696f85560a422f50e8498c"
pipeline = Chronos2Pipeline.from_pretrained(
    "amazon/chronos-2", revision=revision, dtype=torch.float32, device_map="cpu"
)
model = pipeline.model.eval().requires_grad_(False)

def validate(forecast, values, target, weights):
    prediction, gradient, _ = input_attributions(forecast, values)
    quantile_index = list(model.chronos_config.quantiles).index(0.5)
    official = pipeline.predict([values], prediction_length=1)[0][target, quantile_index, 0]
    torch.testing.assert_close(prediction, official)
    for index in [(target, -1), np.unravel_index(weights.abs().argmax(), weights.shape), (0, -32)]:
        delta = torch.zeros_like(values)
        delta[index] = 0.003
        with torch.no_grad():
            finite = (forecast(values + delta) - forecast(values - delta)) / 0.006
        torch.testing.assert_close(gradient[index], finite, atol=0.003, rtol=0.06)
    _, direct, weighted = input_attributions(lambda x: (x * weights).sum(), values)
    torch.testing.assert_close(direct, weights)
    torch.testing.assert_close(weighted, weights * values)

C:\Users\mateu\repos\tsfmxai\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. Como avaliamos

Em cada janela, ordenamos as atribuições em valor absoluto. Se a fórmula tem **k dependências diretas**, selecionamos as **k primeiras** e contamos quantas são corretas. Essa contagem equivale a **k × Recall@k**. No multivariado, um acerto exige o canal e o lag corretos. Em caso de empate no corte, usamos o número esperado de acertos entre as opções empatadas. Fazemos a média das janelas em cada trajetória e depois a média entre trajetórias.

Cada termo da equação tem a forma `(canal de origem, lag, coeficiente)`. Com `Lag` e `Parameter` montamos a expressão e, com `MechanismBinding`, associamos os valores aos seus lugares na expressão. O gerador recebe essa expressão e seus valores.

In [2]:
from tsfmxai.explainability.forecast_gradients import chronos_median, input_attributions, support_scores
from tsfmxai.generators.trajectory import generate_multivariate_trajectory
from tsfmxai.representations.expression import Add, Lag, MechanismBinding, Multiply, Parameter

context_length = 128
seeds = list(range(4210, 4218))
offsets = [0, 32, 64, 96, 128, 160]
innovation_std = 0.35
warmup = 128

def equation(terms):
    # Each term is (source channel, lag, coefficient).
    parameters, lags = {}, {}
    expression = Parameter("zero")
    parameters["zero"] = 0.0
    for index, (source, lag, coefficient) in enumerate(terms):
        name = str(index)
        parameters[name], lags[name] = coefficient, lag
        expression = Add(expression, Multiply(Parameter(name), Lag(source, name)))
    return expression, MechanismBinding(parameters, lags)

def show_formula(equations, targets):
    channel_names = ("X",) if len(equations) == 1 else ("A", "B", "Y")
    lines = []
    for index, terms in enumerate(equations):
        channel = channel_names[index]
        rhs = ""
        for source, lag, coefficient in terms:
            sign = "-" if coefficient < 0 else ("+" if rhs else "")
            rhs += f" {sign} {abs(coefficient):g} {channel_names[source]}_{{t-{lag}}}"
        lines.append(f"{channel}_t &= {rhs} + " + rf"\varepsilon^{{{channel}}}_t" if rhs
                     else f"{channel}_t &= " + rf"\varepsilon^{{{channel}}}_t")
    display(Markdown("$$\n\\begin{aligned}\n" + " \\\\\n".join(lines)
                     + "\n\\end{aligned}\n$$"))
    order = ", ".join(f"{i} = {label}" for i, label in enumerate(channel_names))
    target_names = ", ".join(channel_names[i] for i in targets)
    display(Markdown(
        f"**Canais:** {order}. **Alvo:** {target_names}. "
        f"Cada fórmula vale para as trajetórias com sementes {', '.join(map(str, seeds))}. "
        f"As inovações são gaussianas independentes, com média zero e desvio padrão "
        f"{innovation_std:g}. O histórico inicial é zero e os primeiros {warmup} "
        "passos gerados são descartados."
    ))

scenarios = {}
scenario_results = {}

## 4. Sete consecutivos

### Definir a equação

In [3]:
name = 'Sete consecutivos'
equations = [[(0, 1, 0.035), (0, 2, 0.035), (0, 3, 0.035), (0, 4, 0.035), (0, 5, 0.035), (0, 6, 0.035), (0, 7, 0.035)]]
targets = [0]
scenarios[name] = (equations, targets)
show_formula(equations, targets)

$$
\begin{aligned}
X_t &=   0.035 X_{t-1} + 0.035 X_{t-2} + 0.035 X_{t-3} + 0.035 X_{t-4} + 0.035 X_{t-5} + 0.035 X_{t-6} + 0.035 X_{t-7} + \varepsilon^{X}_t
\end{aligned}
$$

**Canais:** 0 = X. **Alvo:** X. Cada fórmula vale para as trajetórias com sementes 4210, 4211, 4212, 4213, 4214, 4215, 4216, 4217. As inovações são gaussianas independentes, com média zero e desvio padrão 0.35. O histórico inicial é zero e os primeiros 128 passos gerados são descartados.

### Gerar as trajetórias

Passamos as expressões, o histórico inicial e a semente a `generate_multivariate_trajectory`. O retorno tem um canal por linha, inclusive no caso univariado. Recortamos o histórico inicial e o período de aquecimento antes de montar as janelas.

In [4]:
expressions, bindings = zip(*(equation(terms) for terms in equations))
history = max(lag for terms in equations for _, lag, _ in terms)
steps = warmup + offsets[-1] + context_length + 1
trajectories = {}
for seed in seeds:
    series = generate_multivariate_trajectory(
        expressions, bindings, initial_values=np.zeros((len(equations), history)),
        generated_steps=steps, innovation_std=innovation_std, seed=seed,
    )
    trajectories[seed] = series[:, history + warmup:]

### Recuperar as dependências diretas

Para cada janela, `input_attributions` calcula Gradiente e Gradiente × Input sobre a mesma previsão. O suporte de referência vem dos pares canal–lag da equação.

In [5]:
rows = []
for seed, series in trajectories.items():
    for target in targets:
        weights = torch.zeros((len(equations), context_length))
        for source, lag, coefficient in equations[target]:
            weights[source, -lag] = coefficient
        support = weights.numpy() != 0
        k = int(support.sum())
        forecast = partial(chronos_median, model, target=target)
        for offset in offsets:
            stop = offset + context_length
            values = torch.tensor(series[:, offset:stop], dtype=torch.float32)
            if seed == seeds[0] and offset == offsets[0]:
                validate(forecast, values, target, weights)
            _, gradient, weighted = input_attributions(forecast, values)
            for method, attribution in [("Gradiente", gradient), ("Gradiente × Input", weighted)]:
                hits = k * support_scores(attribution.numpy(), support)["recall_at_k"]
                rows.append((seed, target, method, k, hits))
results = pd.DataFrame(rows, columns=["seed", "target", "method", "k", "hits"])
by_trajectory = results.groupby(["seed", "target", "method", "k"], as_index=False).hits.mean()
averages = by_trajectory.groupby(["target", "method", "k"], sort=False).hits.mean()
display(Markdown(
    f"Geramos **{len(seeds)} trajetórias** e avaliamos **{len(offsets)} janelas por trajetória**."
))
for (target, method, k), hits in averages.items():
    channel = ("X",) if len(equations) == 1 else ("A", "B", "Y")
    unit = "lag direto" if k == 1 else "lags diretos"
    if len(equations) > 1:
        unit = "par canal–lag direto" if k == 1 else "pares canal–lag diretos"
    mean = f"{hits:.2f}".replace(".", ",")
    display(Markdown(f"- **{method}**, alvo {channel[target]}: recuperou em média **{mean} de {k} {unit}**."))
scenario_results[name] = averages

Geramos **8 trajetórias** e avaliamos **6 janelas por trajetória**.

- **Gradiente**, alvo X: recuperou em média **3,06 de 7 lags diretos**.

- **Gradiente × Input**, alvo X: recuperou em média **2,12 de 7 lags diretos**.

## 5. Sete esparsos

### Definir a equação

In [6]:
name = 'Sete esparsos'
equations = [[(0, 1, 0.035), (0, 3, 0.035), (0, 6, 0.035), (0, 9, 0.035), (0, 11, 0.035), (0, 13, 0.035), (0, 16, 0.035)]]
targets = [0]
scenarios[name] = (equations, targets)
show_formula(equations, targets)

$$
\begin{aligned}
X_t &=   0.035 X_{t-1} + 0.035 X_{t-3} + 0.035 X_{t-6} + 0.035 X_{t-9} + 0.035 X_{t-11} + 0.035 X_{t-13} + 0.035 X_{t-16} + \varepsilon^{X}_t
\end{aligned}
$$

**Canais:** 0 = X. **Alvo:** X. Cada fórmula vale para as trajetórias com sementes 4210, 4211, 4212, 4213, 4214, 4215, 4216, 4217. As inovações são gaussianas independentes, com média zero e desvio padrão 0.35. O histórico inicial é zero e os primeiros 128 passos gerados são descartados.

### Gerar as trajetórias

Passamos as expressões, o histórico inicial e a semente a `generate_multivariate_trajectory`. O retorno tem um canal por linha, inclusive no caso univariado. Recortamos o histórico inicial e o período de aquecimento antes de montar as janelas.

In [7]:
expressions, bindings = zip(*(equation(terms) for terms in equations))
history = max(lag for terms in equations for _, lag, _ in terms)
steps = warmup + offsets[-1] + context_length + 1
trajectories = {}
for seed in seeds:
    series = generate_multivariate_trajectory(
        expressions, bindings, initial_values=np.zeros((len(equations), history)),
        generated_steps=steps, innovation_std=innovation_std, seed=seed,
    )
    trajectories[seed] = series[:, history + warmup:]

### Recuperar as dependências diretas

Para cada janela, `input_attributions` calcula Gradiente e Gradiente × Input sobre a mesma previsão. O suporte de referência vem dos pares canal–lag da equação.

In [8]:
rows = []
for seed, series in trajectories.items():
    for target in targets:
        weights = torch.zeros((len(equations), context_length))
        for source, lag, coefficient in equations[target]:
            weights[source, -lag] = coefficient
        support = weights.numpy() != 0
        k = int(support.sum())
        forecast = partial(chronos_median, model, target=target)
        for offset in offsets:
            stop = offset + context_length
            values = torch.tensor(series[:, offset:stop], dtype=torch.float32)
            if seed == seeds[0] and offset == offsets[0]:
                validate(forecast, values, target, weights)
            _, gradient, weighted = input_attributions(forecast, values)
            for method, attribution in [("Gradiente", gradient), ("Gradiente × Input", weighted)]:
                hits = k * support_scores(attribution.numpy(), support)["recall_at_k"]
                rows.append((seed, target, method, k, hits))
results = pd.DataFrame(rows, columns=["seed", "target", "method", "k", "hits"])
by_trajectory = results.groupby(["seed", "target", "method", "k"], as_index=False).hits.mean()
averages = by_trajectory.groupby(["target", "method", "k"], sort=False).hits.mean()
display(Markdown(
    f"Geramos **{len(seeds)} trajetórias** e avaliamos **{len(offsets)} janelas por trajetória**."
))
for (target, method, k), hits in averages.items():
    channel = ("X",) if len(equations) == 1 else ("A", "B", "Y")
    unit = "lag direto" if k == 1 else "lags diretos"
    if len(equations) > 1:
        unit = "par canal–lag direto" if k == 1 else "pares canal–lag diretos"
    mean = f"{hits:.2f}".replace(".", ",")
    display(Markdown(f"- **{method}**, alvo {channel[target]}: recuperou em média **{mean} de {k} {unit}**."))
scenario_results[name] = averages

Geramos **8 trajetórias** e avaliamos **6 janelas por trajetória**.

- **Gradiente**, alvo X: recuperou em média **1,85 de 7 lags diretos**.

- **Gradiente × Input**, alvo X: recuperou em média **1,23 de 7 lags diretos**.

### Verificação das equações

Conferimos o limite de propagação definido para os cenários de dependência direta.

In [9]:
# Ganho total < 1. Para uma origem fixa, a soma de todos os caminhos
# de comprimento >= 2 é limitada por max(coef) * ganho / (1 - ganho).
for equations, _ in scenarios.values():
    coefficients = [coef for _, _, coef in equations[0]]
    gain = sum(coefficients)
    assert max(coefficients) * gain / (1 - gain) < min(coefficients)

## 6. Registro do experimento

In [10]:
# Registro completo do experimento
question = 'Os métodos recuperam sete lags conhecidos? Comparamos suportes consecutivos e esparsos, com efeitos diretos equilibrados e um limite analítico para efeitos indiretos.'
reproduction = '$env:PYTHONPATH = (Resolve-Path src).Path; .venv/Scripts/python.exe -m nbconvert --to notebook --execute --inplace --ExecutePreprocessor.timeout=600 experiments/chronos2_explainability/01_chronos2_univariate_direct_dependencies.ipynb'
print(json.dumps({
    "pergunta_e_hipotese": question,
    "checkpoint": ["amazon/chronos-2", revision],
    "base_git": "3c0e09ca20e9c3588af923da6ba928f304994761",
    "codigo": "src/tsfmxai/explainability/forecast_gradients.py e gerador AST do repositório",
    "versoes": {p: version(p) for p in ("chronos-forecasting", "transformers", "torch", "numpy", "pandas", "matplotlib", "nbconvert")},
    "hardware": {"device": "cpu", "threads": 4, "machine": platform.machine(), "processor": platform.processor()},
    "python": platform.python_version(),
    "dtype": "float32",
    "horizonte": 1, "quantil": 0.5, "contexto": context_length,
    "equacoes_e_alvos": scenarios, "seeds_inovacoes": seeds,
    "janelas_inicio_pos_aquecimento": offsets, "aquecimento": warmup,
    "inicializacao": "zeros", "ruido": ["gaussiano independente", innovation_std],
    "identidade": ["scenario", "seed", "offset", "target"],
    "metrica": "Acertos no top-k, com k igual ao número de dependências diretas.",
    "resultados": [
        {"cenario": name, "alvo": target, "metodo": method, "acertos_medios": round(hits, 4), "k": k}
        for name, averages in scenario_results.items()
        for (target, method, k), hits in averages.items()
    ],
    "reproducao": reproduction,
}, ensure_ascii=False, indent=2))
print('A contagem mede recuperação do suporte direto nas equações fixas deste notebook.')

{
  "pergunta_e_hipotese": "Os métodos recuperam sete lags conhecidos? Comparamos suportes consecutivos e esparsos, com efeitos diretos equilibrados e um limite analítico para efeitos indiretos.",
  "checkpoint": [
    "amazon/chronos-2",
    "29ec3766d36d6f73f0696f85560a422f50e8498c"
  ],
  "base_git": "3c0e09ca20e9c3588af923da6ba928f304994761",
  "codigo": "src/tsfmxai/explainability/forecast_gradients.py e gerador AST do repositório",
  "versoes": {
    "chronos-forecasting": "2.2.2",
    "transformers": "4.57.6",
    "torch": "2.12.0",
    "numpy": "1.26.4",
    "pandas": "2.1.4",
    "matplotlib": "3.10.9",
    "nbconvert": "7.17.1"
  },
  "hardware": {
    "device": "cpu",
    "threads": 4,
    "machine": "AMD64",
    "processor": "Intel64 Family 6 Model 158 Stepping 10, GenuineIntel"
  },
  "python": "3.11.9",
  "dtype": "float32",
  "horizonte": 1,
  "quantil": 0.5,
  "contexto": 128,
  "equacoes_e_alvos": {
    "Sete consecutivos": [
      [
        [
          [
            0